In [19]:
import json
import httpx
import openai
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


def get_popular_movies():
    response = httpx.get(f"{BASE_URL}/movies")
    return response.json()


def get_movie_details(id):
    response = httpx.get(f"{BASE_URL}/movies/{id}")
    return response.json()


def get_movie_credits(id):
    response = httpx.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

def get_movie_similar(id):
    response = httpx.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 특정 영화의 상세 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 ID"}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 ID로 출연진 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 ID"}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_similar",
            "description": "영화 ID로 유사한 영화를 가져옵니다..",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 ID"}},
                "required": ["id"],
            },
        },
    },    
]

function_map = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_movie_similar": get_movie_similar,
}


def run_agent(user_message):
    print(f"\n사용자: {user_message}")
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        tools=TOOLS,
        messages=messages,
    )

    message = response.choices[0].message

    if message.tool_calls:
        messages.append(message)

        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f"호출 함수: {func_name}({args})")

            result = function_map[func_name](**args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        print(f"응답: {final_response.choices[0].message.content}")
    else:
        print(f"응답: {message.content}")


# 테스트
run_agent("지금 인기 있는 영화가 무엇인지 알려줘")
run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")
run_agent("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")


사용자: 지금 인기 있는 영화가 무엇인지 알려줘
호출 함수: get_popular_movies({})
응답: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**
   - 개요: 신비로운 "One Wish Willow"를 깨뜨리면서 자신의 사랑을 얻으려는 희망 없는 로맨티스트가 원하는 것을 얻지만, 그 대가가 어두운 것임을 발견하게 됩니다.
   - 개봉일: 2026-05-13
   - 평점: 7.9
   - ![포스터](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**
   - 개요: 1980년대 인도 앤드라 프라데시의 시골에서, 한 열정적인 마을 주민이 스포츠를 통해 공동체를 하나로 모아 강력한 라이벌에 맞섭니다.
   - 개봉일: 2026-06-03
   - 평점: 5.9
   - ![포스터](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**
   - 개요: 결혼을 두고 갈등이 생긴 Jass가 새로운 해외 로맨스를 시작하지만 충격적인 사실로 인해 사랑과 충성심, 약속의 진정한 의미에 직면하게 됩니다.
   - 개봉일: 2026-06-04
   - 평점: 5.1
   - ![포스터](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **Lee Cronin's The Mummy**
   - 개요: 기자의 어린 딸이 사라진 후 8년 만에 돌아오며, 잔인한 가족의 재회가 악몽으로 변하게 됩니다.
   - 개봉일: 2026-04-15
   - 평점: 8.1
   - ![포스터](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)

5. **The Mandalorian and Grog